# Assignment 6B

UAS Change Detection

Corey White  
2026-11-09

<figure>
<a
href="https://colab.research.google.com/github/ncsu-geoforall-lab/gis-584-uas-course/blob/gh-pages/course/topics/topic_6_change_detection/assignments/assignment_6b.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open in Colab</figcaption>
</figure>

## Objective

Apply DEM of Difference and NDVI differencing to provided datasets

Focus on:

- Co-registration and alignment accuracy
- Thresholding and LoD95 calculation
- Quantifying and visualizing uncertainty

In [1]:
# Import standard Python packages we need
import sys
import subprocess
from pathlib import Path

# Ask GRASS where its Python packages are to be able to run it from the notebook
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

In [2]:
# Import the GRASS packages
import grass.script as gs
import grass.jupyter as gj

## Lake Wheeler - Ortho

In [9]:
# Start GRASS in default project mapset
home_directory = Path.home()
grassdata = Path(home_directory, "grassdata", "Lake_Wheeler_NCspm", "assignment6b")
session = gj.init(grassdata)

## Data

Import orthoimagery from Lake Wheeler from flights taken on 07/17/2024
and 09/17/2025.

In [10]:
# Import Lake Wheeler orthophotos into GRASS directly from URLs using r.import
import urllib.parse
import re
from pathlib import Path
import grass.script as gs

orthos = {
    "ortho_2024_07_17": "https://storage.googleapis.com/gis-course-data/gis584/uas-flight-data/Lake%20Wheeler%20-%20NCSU/071724/odm_orthophoto.tif",
    "ortho_2025_09_17": "https://storage.googleapis.com/gis-course-data/gis584/uas-flight-data/Lake%20Wheeler%20-%20NCSU/091725/ortho_2025_09_17.cog.tif",
}


# You can also write this to import the data in parallel
def import_orthos(prefix, url):
    print(f"\nProcessing {prefix}: {url}")

    try:
        gs.run_command("r.import", input=url, output=prefix, overwrite=True)
    except Exception as e:
        print(f"Import failed for {url}: {e}")

    try:
        print("Add semantic labels...")
        bands = [f"{prefix}.{i + 1}" for i in range(3)]
        gs.run_command("r.semantic.label", map=bands, semantic="r,g,b")
    except Exception as e:
        print(f"Semantic labeling failed {e}")


for prefix, url in orthos.items():
    import_orthos(prefix, url)

Check that the images were imported.

In [11]:
gs.run_command("g.list", type="raster", mapset="assignment6b")

Notice that the flights were imported each with four bands.

| Name             | Band | Semantic Label |
|------------------|------|----------------|
| ortho_2024_07_17 | 1    | red            |
| ortho_2024_07_17 | 2    | green          |
| ortho_2024_07_17 | 3    | blue           |
| ortho_2024_07_17 | 4    | alpha          |
| ortho_2025_09_17 | 1    | red            |
| ortho_2025_09_17 | 2    | green          |
| ortho_2025_09_17 | 3    | blue           |
| ortho_2025_09_17 | 4    | alpha          |

Visualize the composite RGB data.

``` python
m = gj.Map(width=600)
m.d_rgb(red="ortho_2024_07_17.1", green="ortho_2024_07_17.2", blue="ortho_2024_07_17.3")
m.d_barscale()
m.show()

m = gj.Map(width=600)
m.d_rgb(red="ortho_2025_09_17.1", green="ortho_2025_09_17.2", blue="ortho_2025_09_17.3")
m.d_barscale()
m.show()
```

Figure 1

Examine the metadata of `ortho_2024_07_17.1` using `r.info`.

In [13]:
gs.run_command("r.info", map="ortho_2024_07_17.1")

In [14]:
gs.run_command("r.info", map="ortho_2025_09_17.1")

> **Important**
>
> What is the spatial resolution of the orthoimagery.

View the data as an interactive webmap.

``` python
m = gj.InteractiveMap(use_region=False)
m.add_raster("ortho_2024_07_17.1", opacity=0.6)
m.show()
```

Figure 2

Set the computational region to $0.07 m$ with a spatial extent focused
on a single field using the following command.

In [16]:
gs.run_command(
    "g.region", n=219488, s=219104, e=637233, w=636830, res="0.07", flags="pa"
)

View the historams of the red bands.

In [17]:
from grass.script import array as garray
import seaborn as sns
import matplotlib.pyplot as plt

red_25 = garray.array(mapname="ortho_2025_09_17.1")
red_24 = garray.array(mapname="ortho_2024_07_17.1")
# Plot raster histogram
sns.histplot(data=red_25.ravel(), kde=True)
sns.histplot(data=red_24.ravel(), kde=True)
plt.show()

In [18]:
m = gj.Map(width=600, use_region=True)
m.d_rgb(red="ortho_2024_07_17.1", green="ortho_2024_07_17.2", blue="ortho_2024_07_17.3")
m.d_barscale()
m.show()

m = gj.Map(width=600, use_region=True)
m.d_rgb(red="ortho_2025_09_17.1", green="ortho_2025_09_17.2", blue="ortho_2025_09_17.3")
m.d_barscale()
m.show()

In [19]:
gs.run_command(
    "r.composite",
    red="ortho_2025_09_17.1",
    green="ortho_2025_09_17.2",
    blue="ortho_2025_09_17.3",
    output="ortho_2025_09_17.rgb",
)

In [20]:
gs.run_command("r.univar", map="ortho_2024_07_17.1", flags="e")

In [21]:
gs.run_command("r.univar", map="ortho_2025_09_17.1", flags="e")

### VARI

In [22]:
gs.run_command(
    "r.mapcalc",
    expression="vari_2024 = ((float(ortho_2024_07_17.2) / 255.0) - (float(ortho_2024_07_17.1) / 255.0)) / ((float(ortho_2024_07_17.2) / 255.0) + (float(ortho_2024_07_17.1) / 255.0) - (float(ortho_2024_07_17.3) / 255.0) + 0.175)",
)

gs.run_command("r.univar", map="vari_2024", flags="e")

In [23]:
gs.run_command("r.colors", map="vari_2024", color="ndvi", flags="")
m = gj.Map(width=600, use_region=True)
m.d_rast(map="vari_2024")
m.d_legend(raster="vari_2024")
m.d_barscale()
m.show()

In [24]:
gs.run_command(
    "r.mapcalc",
    expression="vari_2025 = ((float(ortho_2025_09_17.2) / 255.0) - (float(ortho_2025_09_17.1) / 255.0)) / ((float(ortho_2025_09_17.2) / 255.0) + (float(ortho_2025_09_17.1) / 255.0) - (float(ortho_2025_09_17.3) / 255.0) + 0.175)",
)

gs.run_command("r.univar", map="vari_2025", flags="e")

In [25]:
gs.run_command("r.colors", map="vari_2025", color="ndvi", flags="")
m = gj.Map(width=600, use_region=True)
m.d_rast(map="vari_2025")
m.d_legend(raster="vari_2025")
m.d_barscale()
m.show()

In [26]:
gs.run_command(
    "r.mapcalc",
    expression="vari_diff = vari_2025 - vari_2024",
)

gs.run_command("r.univar", map="vari_diff", flags="e")

In [27]:
gs.run_command("r.colors", map="vari_diff", color="difference", flags="")
m = gj.Map(width=600, use_region=True)
m.d_rast(map="vari_diff")
m.d_legend(raster="vari_diff")
m.d_barscale()
m.show()

## Redness Index

Calculate redness index

$$
RI = \frac{Red^2}{Blue*Green^2}
$$

In [28]:
def ri(r, g, b, output):
    gs.run_command(
        "r.mapcalc",
        expression=f"{output} = if(1.0 *(pow ( {r}, 2 ) )/( {b} * pow( {g} ,2)) >= 0.02, null(), 1.0 * (pow ( {r}, 2 ) )/( {b} * pow( {g} ,2)))",
    )
    return output

In [29]:
ri(
    "ortho_2025_09_17.1",
    "ortho_2025_09_17.2",
    "ortho_2025_09_17.3",
    "ortho_2025_09_17.ri",
)
gs.run_command("r.colors", map="ortho_2025_09_17.ri", color="gyr", flags="e")
m = gj.Map(width=600, use_region=True)
m.d_rast(map="ortho_2025_09_17.ri")
m.d_legend(raster="ortho_2025_09_17.ri", flags="d")
m.d_barscale()
m.show()

In [30]:
ri(
    "ortho_2024_07_17.1",
    "ortho_2024_07_17.2",
    "ortho_2024_07_17.3",
    "ortho_2024_07_17.ri",
)
gs.run_command("r.colors", map="ortho_2024_07_17.ri", color="magma", flags="e")
m = gj.Map(width=600, use_region=True)
m.d_rast(map="ortho_2024_07_17.ri")
m.d_legend(raster="ortho_2024_07_17.ri", flags="d")
m.d_barscale()
m.show()

## Visible Vegetation Index (VVI)

$$
VVI = (1 - \frac{Red - 30}{Red + 30})(1 - \frac{Green - 50}{Green + 50})(1 - \frac{Blue - 1}{Blue + 1})
$$

In [31]:
def vvi(r, g, b, output):
    """
    Visible Vegetation Index (VVI)
    """
    expression = f"{output} = (1.0 - ((float({r}) / 255.0 - 30.0) / (float({r}) / 255.0 + 30.0 + 0.000001))) * (1.0 - ((float({g}) / 255.0 - 50.0) / (float({g}) / 255.0 + 50.0 + 0.000001))) * (1.0 - ((float({b}) / 255.0 - 1.0) / (float({b}) / 255.0 + 1.0 + 0.000001)))"
    gs.run_command("r.mapcalc", expression=expression)
    return output

In [32]:
vvi(
    r="ortho_2024_07_17.1",
    g="ortho_2024_07_17.2",
    b="ortho_2024_07_17.3",
    output="ortho_2024_07_17.vvi",
)
gs.run_command("r.colors", map="ortho_2024_07_17.vvi", color="magma", flags="e")
m = gj.Map(width=600, use_region=True)
m.d_rast(map="ortho_2024_07_17.vvi")
m.d_legend(raster="ortho_2024_07_17.vvi", flags="db")
m.d_barscale()
m.show()

## Brightness Index (BI)

$$
BI = \frac{\sqrt{Red^2 + Green^2 + Blue^2}}{3}
$$

In [33]:
def bi(r, g, b, output):
    """
    Brightness Index (BI)
    """
    expression = f"{output} = sqrt(pow(float({r}),2)  + pow(float({g}),2) + pow(float({b}),2)) / 3.0"
    gs.run_command("r.mapcalc", expression=expression)
    return output

In [34]:
bi(
    r="ortho_2024_07_17.1",
    g="ortho_2024_07_17.2",
    b="ortho_2024_07_17.3",
    output="ortho_2024_07_17.bi",
)
gs.run_command("r.colors", map="ortho_2024_07_17.bi", color="wave", flags="")
m = gj.Map(width=600, use_region=True)
m.d_rast(map="ortho_2024_07_17.bi")
m.d_legend(raster="ortho_2024_07_17.bi", flags="d")
m.d_barscale()
m.show()

## Spectral Slope Saturation Index (SI)

$$
SI = \frac{(Red - Blue)}{(Red + Blue)}
$$

In [35]:
def si(r, b, output):
    """
    Compute SI = (R - B) / (R + B) with protection against division by zero.
    r, b: raster names (band maps)
    output: name of output raster
    """
    expr = (
        f"{output} = if( (float({r}) + float({b})) == 0, null(), "
        f"(float({r}) - float({b})) / (float({r}) + float({b}) + 0.00000001) )"
    )
    gs.run_command("r.mapcalc", expression=expr, overwrite=True)
    return output

In [36]:
si(
    r="ortho_2024_07_17.1",
    b="ortho_2024_07_17.3",
    output="ortho_2024_07_17.si",
)
gs.run_command("r.colors", map="ortho_2024_07_17.si", color="viridis", flags="e")
m = gj.Map(width=600, use_region=True)
m.d_rast(map="ortho_2024_07_17.si")
m.d_legend(raster="ortho_2024_07_17.si", flags="db")
m.d_barscale()
m.show()